# T2.5 파일럿 베이스라인 1시드 학습 + 진단 4종 (Colab T4)

역할: **브랜치 가져오기 → 데이터 해제 → `scripts/train_pilot_baseline.py` → `scripts/run_pilot_diagnosis.py` → `results/pilot/` 커밋**.
로직은 `src/sr_pipeline/`에 있고 이 노트북에는 넣지 않습니다.

- 이 학습은 **진단용 파일럿**입니다. 결과 표·집계에 넣지 않고 `results/pilot/`에만 둡니다.
- 에폭·에폭당 패치·배치·학습률은 `pilot_determined`가 `null`인 동안 코드에 기본값이 없습니다. 셀 1에서 정해 넣으세요(`None`이면 중단).
- 체크포인트(`*.pth`)는 Drive에 두고 커밋하지 않습니다. 저장소가 Public이라 링크·토큰·경로를 출력이나 커밋에 남기지 않습니다.

In [ ]:
# 1. 실행 파라미터 (본인 값으로 채운다. None이면 아래 셀이 중단한다)
REPO_URL = "https://github.com/minyi-k03/VideoProcessing-Project.git"
BRANCH = "main"            # 본인 이름 브랜치 또는 main
SEED = 0                   # 파일럿은 1시드
EPOCHS = None
PATCHES_PER_EPOCH = None
BATCH_SIZE = None
LEARNING_RATE = None
EDGE_TOP_Q = None          # 에지 PSNR 상위 비율 q(%), 확정 전 파일럿 값
DATA_ZIP = None            # 예: /content/drive/MyDrive/<폴더>/<patch_pool_id>.zip
DATA_DIR = "/content/data"
CKPT_PATH = None           # 예: /content/drive/MyDrive/<폴더>/pilot_baseline_s0.pth (Drive, 커밋 안 함)
SR_DIR = "/content/pilot_sr"
missing = [k for k, v in dict(EPOCHS=EPOCHS, PATCHES_PER_EPOCH=PATCHES_PER_EPOCH, BATCH_SIZE=BATCH_SIZE,
           LEARNING_RATE=LEARNING_RATE, EDGE_TOP_Q=EDGE_TOP_Q, DATA_ZIP=DATA_ZIP, CKPT_PATH=CKPT_PATH).items() if v is None]
assert not missing, f"값을 채우세요: {missing}"

In [ ]:
# 2. 브랜치 가져오기
!git clone -b {BRANCH} {REPO_URL} /content/VideoProcessing-Project
%cd /content/VideoProcessing-Project
!pip install -q -r requirements.txt

In [ ]:
# 3. 데이터 해제 (Drive 마운트 후 압축 파일 하나를 로컬로 푼다)
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p {DATA_DIR}
!unzip -q -o {DATA_ZIP} -d {DATA_DIR}

In [ ]:
# 4. 학습 + SR 출력 생성 (T4). 학습 설정은 results/pilot/pilot_train_settings.json에 기록된다.
!PYTHONPATH=src python scripts/train_pilot_baseline.py --data-dir {DATA_DIR} --checkpoint-path {CKPT_PATH} --sr-dir {SR_DIR} --output-dir results/pilot --seed {SEED} --epochs {EPOCHS} --patches-per-epoch {PATCHES_PER_EPOCH} --batch-size {BATCH_SIZE} --learning-rate {LEARNING_RATE}

In [ ]:
# 5. 진단 4종 (오차 지도, 에지·평탄 PSNR(Y), 주파수 스펙트럼, YCbCr 채널별 오차 + 최악 영상)
!PYTHONPATH=src python scripts/run_pilot_diagnosis.py --sr-dir {SR_DIR} --hr-dir {DATA_DIR}/testsets --output-dir results/pilot --edge-top-q {EDGE_TOP_Q}

In [ ]:
# 6. results/pilot/ 커밋·push (토큰은 실행할 때만 입력하고 어디에도 남기지 않는다)
!git status --short results/pilot
!git add results/pilot
!git commit -m "results: T2.5 pilot baseline seed {SEED} diagnosis"
import getpass, subprocess
token = getpass.getpass("GitHub token (not shown): ")
# 실패해도 토큰이 든 URL이 셀 출력에 찍히지 않도록 출력을 캡처하고 종료 코드만 보여 준다.
result = subprocess.run(["git", "push", f"https://{token}@github.com/minyi-k03/VideoProcessing-Project.git", f"HEAD:{BRANCH}"],
                        capture_output=True, text=True)
del token
print("push exit code:", result.returncode)
if result.returncode != 0:
    print("push 실패: 토큰 권한과 브랜치 이름을 확인하세요(자세한 출력은 토큰 노출 위험으로 숨김).")